# NB-1 — D-NORM-1: DeBERTa intervention decomposition (frozen backbone)

**Evaluation only. No training, no checkpoint modification, no internal-validation runs.**

Implements Part A of `FMI_manifests_and_decision_rules_v1_2.md` for the nine frozen-backbone
checkpoints (LRAA/`lcca`, token-wise MLP, token mixer × seeds 13, 42, 71):

- three intervention states on the same checkpoint — `on`, `norm_only` (Z = 0, trained LayerNorm kept), `off` (module bypassed);
- decomposition `on − off = (on − norm_only) + (norm_only − off)` per seed, with seed-level t intervals and a
  passage-cluster bootstrap (10,000 replicates, seed 20260924);
- D4 descriptives: learned gate value and mean relative branch norm ‖g·Z[CLS]‖ / ‖H[CLS]‖;
- gates A.7.1–A.7.3 (checkpoint hashes; label-exact parity against the archived per-example predictions where they
  exist; aggregate parity where only aggregates exist; A + B = archived on − off within 0.01 point).

If any gate fails the notebook raises **after** saving the raw predictions, so the deviation can be examined.
The held-out set is the official BoolQ validation split (3,270 rows); these checkpoints were already reported on it.

Inputs: the public repository (cloned automatically) and the base `deberta-v3-large` folder on Google Drive.
Output: a result ZIP downloaded automatically and copied to `MyDrive/FMI/D-NORM-1/`. The runtime is not deleted.

In [ ]:
# 1) Mount Google Drive first; record the start time.
import time
from datetime import datetime, timezone
RUN_STARTED_PERF = time.perf_counter()
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
from google.colab import drive
drive.mount('/content/drive')
print('Run started (UTC):', RUN_STARTED_UTC)

In [ ]:
# 2) Pinned dependencies (same as the archived adapter-off audit notebook).
import subprocess, sys
packages = ['transformers==4.57.6', 'huggingface-hub==0.36.2', 'datasets==3.2.0',
            'pandas==2.2.3', 'scikit-learn==1.6.1', 'sentencepiece', 'protobuf']
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], text=True, capture_output=True)
if proc.returncode != 0:
    print(proc.stdout); print(proc.stderr); raise RuntimeError('Dependency installation failed.')
print('Dependencies ready.')

In [ ]:
# 3) Imports, GPU audit, paths, local copy of the base model.
import gc, hashlib, json, math, os, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required (Runtime > Change runtime type > GPU). A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

MANIFEST_ID = 'FMI_manifests_and_decision_rules_v1_2.md / Part A (D-NORM-1)'
MANIFEST_LOCK_COMMIT = ''   # fill in from MANIFEST_LOCK.md before running; recorded, not checked
BOOTSTRAP_SEED = 20260924
BOOTSTRAP_REPLICATES = 10000

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
WORK_DIR = Path('/content/fmi_dnorm1'); REPO_DIR = WORK_DIR / 'repo'; RESULT_DIR = WORK_DIR / 'results'
DRIVE_OUT_DIR = Path('/content/drive/MyDrive/FMI/D-NORM-1')
for p in (WORK_DIR, RESULT_DIR, DRIVE_OUT_DIR): p.mkdir(parents=True, exist_ok=True)

required = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [n for n in required if not (DRIVE_MODEL_DIR / n).is_file()]
has_weights = any((DRIVE_MODEL_DIR / n).is_file() for n in ('model.safetensors', 'pytorch_model.bin'))
if missing or not has_weights:
    raise FileNotFoundError(f'Incomplete base model folder {DRIVE_MODEL_DIR}: missing={missing}, weights={has_weights}')
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME, '| torch', torch.__version__, '| CUDA', torch.version.cuda)

In [ ]:
# 4) Clone the public artifact repository; verify the twelve checkpoint hashes (gate A.7.1); locate archives.
REPO_URL = 'https://github.com/kurbankotan/Adapter-Contribution-Audit.git'
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
if r.returncode != 0:
    print(r.stderr); raise RuntimeError('git clone failed')
REPO_COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True, capture_output=True).stdout.strip()
print('Repository commit:', REPO_COMMIT)

def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def find_one(pattern):
    hits = sorted(REPO_DIR.rglob(pattern), key=lambda p: len(p.parts))
    if not hits: raise FileNotFoundError(f'Not found in repository: {pattern}')
    return hits[0]

# 4a) frozen V9 checkpoints (head, mlp, lcca) — manifest columns: model, seed, filename, bytes, sha256
V9_MANIFEST = find_one('checkpoint_manifest.csv')
v9_ckpt_dir = V9_MANIFEST.parent if V9_MANIFEST.parent.name == 'checkpoints' else V9_MANIFEST.parent / 'checkpoints'
v9m = pd.read_csv(V9_MANIFEST)
CKPT = {}
for row in v9m.itertuples(index=False):
    p = v9_ckpt_dir / row.filename
    if not p.is_file(): raise FileNotFoundError(p)
    if p.stat().st_size != int(row.bytes) or sha256_file(p) != row.sha256:
        raise RuntimeError(f'HASH MISMATCH: {row.filename}')
    CKPT[(str(row.model), int(row.seed))] = p
# 4b) V13 token-mixer checkpoints — manifest: checkpoint_manifest_sha256.csv in results/v13_token_mixer_lora_controls
V13_MANIFEST = find_one('checkpoint_manifest_sha256.csv')
v13m = pd.read_csv(V13_MANIFEST)
fname_col = next(c for c in v13m.columns if 'file' in c.lower() or 'name' in c.lower())
sha_col = next(c for c in v13m.columns if 'sha' in c.lower())
bytes_col = next((c for c in v13m.columns if 'byte' in c.lower() or 'size' in c.lower()), None)
for row in v13m.to_dict('records'):
    fname = Path(str(row[fname_col])).name
    if not fname.startswith('token_mixer'): continue
    p = find_one(fname)
    if sha256_file(p) != row[sha_col] or (bytes_col and p.stat().st_size != int(row[bytes_col])):
        raise RuntimeError(f'HASH MISMATCH: {fname}')
    seed = int(fname.split('seed')[1].split('.')[0])
    CKPT[('token_mixer', seed)] = p
needed = {(m, s) for m in ('lcca', 'mlp', 'token_mixer') for s in (13, 42, 71)}
if not needed.issubset(CKPT): raise RuntimeError(f'Missing checkpoints: {needed - set(CKPT)}')
print('Gate A.7.1 PASS — checkpoint hashes verified:', len(needed), 'used of', len(CKPT), 'listed')

# 4c) archives used by the parity gates (A.7.2)
ARCH_V9_RUNS = find_one('table_run_results.csv')                       # archived V9 held-out aggregates (incl. head-only)
ARCH_ONOFF_PRED = find_one('adapter_on_off_predictions.csv')           # LRAA/MLP per-example on/off labels
ARCH_V13_PRED = find_one('heldout_predictions.csv')                    # token mixer/LoRA per-example on labels + prob_yes
ARCH_V13_RUNS = find_one('control_run_results.csv')                    # token mixer/LoRA aggregates incl. adapter_off_*
for p in (ARCH_V9_RUNS, ARCH_ONOFF_PRED, ARCH_V13_PRED, ARCH_V13_RUNS): print('archive:', p.relative_to(REPO_DIR))
ARCHIVE_HASHES = {p.name: sha256_file(p) for p in (V9_MANIFEST, V13_MANIFEST, ARCH_V9_RUNS, ARCH_ONOFF_PRED, ARCH_V13_PRED, ARCH_V13_RUNS)}

In [ ]:
# 5) Official BoolQ validation split, passage clusters, exact serialization; two padding modes.
SEEDS = (13, 42, 71); MAX_LENGTH = 512; EVAL_BATCH_SIZE = 4
LATENT_RANK, LATENT_HEADS, MLP_BOTTLENECK = 256, 8, 512
MIXER_RANK, MIXER_KERNEL, MIXER_DILATION = 496, 32, 16
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

heldout = load_dataset('google/boolq', split='validation', token=HF_TOKEN)
if len(heldout) != 3270: raise ValueError(f'Expected 3,270 rows, found {len(heldout)}')
PASSAGE_HASH = [hashlib.sha256(p.encode('utf-8')).hexdigest() for p in heldout['passage']]
_cl = {h: i for i, h in enumerate(sorted(set(PASSAGE_HASH)))}
CLUSTER = np.array([_cl[h] for h in PASSAGE_HASH]); N_CLUSTERS = len(_cl)
GOLD = np.array([int(bool(a)) for a in heldout['answer']])
print('rows', len(GOLD), '| passage clusters', N_CLUSTERS, '| yes rate', round(GOLD.mean() * 100, 2))

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, token=HF_TOKEN, use_fast=True)
def encode_batch(batch):
    prefixes = [f'question: {q} answer:' for q in batch['question']]
    yes = tokenizer([q + ' yes' for q in prefixes], batch['passage'], truncation='only_second', max_length=MAX_LENGTH)
    no = tokenizer([q + ' no' for q in prefixes], batch['passage'], truncation='only_second', max_length=MAX_LENGTH)
    return {'yes_input_ids': yes['input_ids'], 'yes_attention_mask': yes['attention_mask'],
            'no_input_ids': no['input_ids'], 'no_attention_mask': no['attention_mask']}
encoded = heldout.map(encode_batch, batched=True, batch_size=256, remove_columns=['question', 'passage'], desc='tokenize')

class PairDataset(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        r = self.ds[int(i)]
        return {'candidates': [{'input_ids': r['yes_input_ids'], 'attention_mask': r['yes_attention_mask']},
                               {'input_ids': r['no_input_ids'], 'attention_mask': r['no_attention_mask']}],
                'label': 0 if bool(r['answer']) else 1, 'gold': int(bool(r['answer'])), 'index': int(i)}
def make_collate(padding):
    def collate(batch):
        flat = [c for item in batch for c in item['candidates']]
        if padding == 'dynamic': padded = tokenizer.pad(flat, padding=True, return_tensors='pt')
        else: padded = tokenizer.pad(flat, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
        bsz = len(batch)
        return {'input_ids': padded['input_ids'].view(bsz, 2, -1), 'attention_mask': padded['attention_mask'].view(bsz, 2, -1),
                'labels': torch.tensor([it['label'] for it in batch]), 'gold': torch.tensor([it['gold'] for it in batch]),
                'index': torch.tensor([it['index'] for it in batch])}
    return collate
LOADERS = {pad: DataLoader(PairDataset(encoded), batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=make_collate(pad), num_workers=2, pin_memory=True)
           for pad in ('dynamic', 'max_length')}
print('Loaders ready: dynamic padding (V9 adapter-off archive setting) and max_length padding (V13 archive setting).')

In [ ]:
# 6) Exact module definitions (V9 MLP/LRAA from the archived audit notebook; V13 token mixer) with three intervention states.
STATES = ('on', 'norm_only', 'off')

class GatedMLPAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck, bias=False); self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size); self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.alpha_logit)
    def branch(self, hidden, attention_mask): return self.up(F.gelu(self.down(hidden)))
    def forward(self, hidden, attention_mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, attention_mask)))

class LatentRelationAdapter(nn.Module):   # LRAA (legacy id lcca)
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__(); assert rank % heads == 0
        self.rank, self.heads, self.head_dim = rank, heads, rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False); self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False); self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size); self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.alpha_logit)
    def branch(self, hidden, attention_mask):
        bsz, seq_len, _ = hidden.shape
        split = lambda x: x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(attention_mask[:, None, None, :] == 0, torch.finfo(scores.dtype).min)
        z = torch.matmul(torch.softmax(scores, dim=-1), v).transpose(1, 2).contiguous().view(bsz, seq_len, self.rank)
        return self.out(z)
    def forward(self, hidden, attention_mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, attention_mask)))

class DilatedConvMixerAdapter(nn.Module):  # V13 non-attention token mixer (exact)
    def __init__(self, hidden=1024, rank=MIXER_RANK, kernel=MIXER_KERNEL, dilation=MIXER_DILATION):
        super().__init__()
        self.depthwise = nn.Conv1d(hidden, hidden, kernel_size=kernel, dilation=dilation, groups=hidden, bias=False)
        self.down = nn.Linear(hidden, rank, bias=False); self.up = nn.Linear(rank, hidden, bias=False)
        self.norm = nn.LayerNorm(hidden); self.dropout = nn.Dropout(0.10)
        self.gate_param = nn.Parameter(torch.zeros(()))
        self.total_pad = dilation * (kernel - 1)
    @property
    def gate(self): return torch.tanh(self.gate_param)
    def branch(self, hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype); x = hidden * mask
        left = self.total_pad // 2; right = self.total_pad - left
        mixed = self.depthwise(F.pad(x.transpose(1, 2), (left, right))).transpose(1, 2) * mask
        return self.up(F.gelu(self.down(mixed)))
    def forward(self, hidden, attention_mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.gate * self.dropout(self.branch(hidden, attention_mask)))

class FrozenFeatureScorer(nn.Module):
    def __init__(self, mode, hidden_size):
        super().__init__()
        if mode == 'mlp': self.adapter = GatedMLPAdapter(hidden_size, MLP_BOTTLENECK)
        elif mode == 'lcca': self.adapter = LatentRelationAdapter(hidden_size, LATENT_RANK, LATENT_HEADS)
        elif mode == 'token_mixer': self.adapter = DilatedConvMixerAdapter(hidden_size)
        else: raise ValueError(mode)
        self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(hidden_size, 1)
    def forward(self, hidden, attention_mask, state='on'):
        return self.scorer(self.dropout(self.adapter(hidden, attention_mask, state=state)[:, 0]))
    @torch.no_grad()
    def branch_ratio_cls(self, hidden, attention_mask):
        z = self.adapter.branch(hidden, attention_mask)[:, 0].float() * self.adapter.gate.float()
        return (z.norm(dim=-1) / hidden[:, 0].float().norm(dim=-1).clamp_min(1e-6))

def safe_load(p):
    # own repository artifacts, hash-verified above; fall back to full unpickling only if the safe loader rejects a payload field
    try: return torch.load(p, map_location='cpu', weights_only=True)
    except Exception: return torch.load(p, map_location='cpu', weights_only=False)

encoder = AutoModel.from_pretrained(LOCAL_MODEL_DIR, token=HF_TOKEN).to(DEVICE).eval()
for prm in encoder.parameters(): prm.requires_grad = False
HIDDEN = int(encoder.config.hidden_size)

MODELS = {}
for (alias, seed), path in sorted(CKPT.items()):
    if alias not in ('mlp', 'lcca', 'token_mixer'): continue
    ck = safe_load(path); m = FrozenFeatureScorer(alias, HIDDEN).to(DEVICE)
    if alias == 'token_mixer':
        sd = ck['model_state_dict']
        sd = {k.replace('adapter.gate', 'adapter.gate_param') if k == 'adapter.gate' else k: v for k, v in sd.items()}
        m.load_state_dict(sd, strict=True)
    else:
        m.adapter.load_state_dict(ck['adapter_state_dict'], strict=True); m.scorer.load_state_dict(ck['scorer_state_dict'], strict=True)
    m.eval(); MODELS[(alias, seed)] = m
counts = {a: sum(p.numel() for p in MODELS[(a, 13)].adapter.parameters()) for a in ('mlp', 'lcca', 'token_mixer')}
print('Loaded 9 models. Adapter parameter counts:', counts)
if len(set(counts.values())) != 1 or list(counts.values())[0] != 1050625: raise RuntimeError('Parameter match check failed')

In [ ]:
# 7) Two evaluation passes, each in the setting of the archive it must match:
#    pass A: LRAA + MLP, dynamic padding, autocast dtype as in the archived adapter-off audit (bf16 if supported else fp16)
#    pass B: token mixer, max_length padding, bf16 autocast as in the archived V13 evaluation.
PASS_CFG = {
    'A': {'aliases': ('lcca', 'mlp'), 'padding': 'dynamic', 'dtype': torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16},
    'B': {'aliases': ('token_mixer',), 'padding': 'max_length', 'dtype': torch.bfloat16},
}
records = {}   # (alias, seed, state) -> dict(pred, prob_yes, loss_sum)
branch_ratio = {}  # (alias, seed) -> list of per-sequence ratios
t0 = time.perf_counter()
with torch.no_grad():
    for pname, cfg in PASS_CFG.items():
        loader = LOADERS[cfg['padding']]
        keys = [(a, s, st) for a in cfg['aliases'] for s in SEEDS for st in STATES]
        for k in keys: records[k] = {'pred': np.zeros(len(GOLD), dtype=int), 'prob_yes': np.zeros(len(GOLD)), 'loss_sum': 0.0}
        for a in cfg['aliases']:
            for s in SEEDS: branch_ratio[(a, s)] = []
        for bi, batch in enumerate(loader, 1):
            ids = batch['input_ids'].to(DEVICE, non_blocking=True); mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            labels = batch['labels'].to(DEVICE); idx = batch['index'].numpy(); bsz, nopt, L = ids.shape
            flat_ids, flat_mask = ids.view(bsz * nopt, L), mask.view(bsz * nopt, L)
            with torch.amp.autocast('cuda', dtype=cfg['dtype']):
                hidden = encoder(input_ids=flat_ids, attention_mask=flat_mask).last_hidden_state
                for a in cfg['aliases']:
                    for s in SEEDS:
                        model = MODELS[(a, s)]
                        for st in STATES:
                            logits = model(hidden, flat_mask, state=st).view(bsz, nopt).float()
                            p = torch.softmax(logits, dim=-1)[:, 0]
                            rec = records[(a, s, st)]
                            rec['pred'][idx] = (logits.argmax(dim=-1) == 0).long().cpu().numpy()
                            rec['prob_yes'][idx] = p.cpu().numpy()
                            rec['loss_sum'] += F.cross_entropy(logits, labels, reduction='sum').item()
                        branch_ratio[(a, s)].extend(model.branch_ratio_cls(hidden, flat_mask).cpu().numpy().tolist())
            if bi == 1 or bi % 200 == 0 or bi == len(loader): print(f'pass {pname}: {bi}/{len(loader)}')
        gc.collect(); torch.cuda.empty_cache()
print('Evaluation minutes:', round((time.perf_counter() - t0) / 60, 1))

def metrics(pred, prob):
    return {'accuracy': accuracy_score(GOLD, pred), 'balanced_accuracy': balanced_accuracy_score(GOLD, pred),
            'macro_f1': f1_score(GOLD, pred, average='macro', zero_division=0), 'predicted_yes_rate': float(pred.mean()),
            'log_loss': float(log_loss(GOLD, np.clip(prob, 1e-7, 1 - 1e-7)))}
run_rows, pred_rows = [], []
for (a, s, st), rec in records.items():
    mrow = metrics(rec['pred'], rec['prob_yes']); run_rows.append({'model': a, 'seed': s, 'state': st, **mrow})
    for i in range(len(GOLD)):
        pred_rows.append({'model': a, 'seed': s, 'state': st, 'example_index': i, 'passage_cluster': int(CLUSTER[i]),
                          'gold': int(GOLD[i]), 'prediction': int(rec['pred'][i]), 'prob_yes': float(rec['prob_yes'][i])})
RUN = pd.DataFrame(run_rows).sort_values(['model', 'seed', 'state']).reset_index(drop=True)
PRED = pd.DataFrame(pred_rows)
RUN.to_csv(RESULT_DIR / 'three_state_run_results.csv', index=False)
PRED.to_csv(RESULT_DIR / 'three_state_predictions.csv', index=False)
_v9 = pd.read_csv(ARCH_V9_RUNS); _alpha = {(m.replace('_adapter', ''), int(s)): float(v) for m, s, v in zip(_v9.model, _v9.seed, _v9.adapter_alpha) if str(m) != 'head_only'}
D4 = pd.DataFrame([{'model': a, 'seed': s, 'gate_value': float(MODELS[(a, s)].adapter.gate.detach().cpu()),
                    'archived_adapter_alpha': _alpha.get((a, s)),
                    'gate_parameterization': 'tanh' if a == 'token_mixer' else 'sigmoid',
                    'mean_relative_branch_norm_cls': float(np.mean(branch_ratio[(a, s)])),
                    'median_relative_branch_norm_cls': float(np.median(branch_ratio[(a, s)]))} for (a, s) in sorted(branch_ratio)])
D4.to_csv(RESULT_DIR / 'd4_gate_and_branch_norm.csv', index=False)
display(RUN.pivot_table(index=['model', 'seed'], columns='state', values='balanced_accuracy').mul(100).round(2))
display(D4)

In [ ]:
# 8) Parity gates per manifest A.7.2 and A.7.3. Raw predictions are already saved; a failure raises here.
tol_prob = 1e-3
parity = []
def rec_pred(a, s, st): return records[(a, s, st)]['pred']
# LRAA/MLP: label-exact on and off against the archived adapter-off audit predictions
arch = pd.read_csv(ARCH_ONOFF_PRED)
for a in ('lcca', 'mlp'):
    for s in SEEDS:
        for st in ('on', 'off'):
            sub = arch[(arch.model == a) & (arch.seed == s) & (arch.adapter_state == st)].sort_values('example_index')
            if len(sub) != len(GOLD): raise RuntimeError(f'archived predictions incomplete for {a} {s} {st}: {len(sub)}')
            mism = int((sub.prediction.to_numpy() != rec_pred(a, s, st)).sum())
            parity.append({'model': a, 'seed': s, 'state': st, 'check': 'label_exact_vs_adapter_off_archive', 'label_mismatches': mism,
                           'max_abs_prob_dev': None, 'pass': mism == 0})
# token mixer on: labels exact and prob_yes within tolerance against the V13 held-out predictions
v13p = pd.read_csv(ARCH_V13_PRED)
for s in SEEDS:
    sub = v13p[(v13p.model == 'token_mixer') & (v13p.seed == s)].sort_values('example_index')
    if len(sub) != len(GOLD): raise RuntimeError(f'V13 archived predictions incomplete for token_mixer {s}: {len(sub)}')
    mism = int((sub.prediction.to_numpy() != rec_pred('token_mixer', s, 'on')).sum())
    dev = float(np.abs(sub.prob_yes.to_numpy() - records[('token_mixer', s, 'on')]['prob_yes']).max())
    parity.append({'model': 'token_mixer', 'seed': s, 'state': 'on', 'check': 'label_exact_and_prob_vs_v13_archive', 'label_mismatches': mism,
                   'max_abs_prob_dev': dev, 'pass': (mism == 0) and (dev <= tol_prob)})
# token mixer off: aggregates within 0.01 point against control_run_results.csv (adapter_off_*)
v13r = pd.read_csv(ARCH_V13_RUNS)
for s in SEEDS:
    row = v13r[(v13r.model == 'token_mixer') & (v13r.seed == s)].iloc[0]
    mine = RUN[(RUN.model == 'token_mixer') & (RUN.seed == s) & (RUN.state == 'off')].iloc[0]
    devs = {m: 100 * abs(float(row[f'adapter_off_{m}']) - float(mine[m])) for m in ('accuracy', 'balanced_accuracy', 'macro_f1')}
    parity.append({'model': 'token_mixer', 'seed': s, 'state': 'off', 'check': 'aggregates_within_0.01pt_vs_v13_archive',
                   'label_mismatches': None, 'max_abs_prob_dev': None, 'aggregate_devs_points': json.dumps(devs), 'pass': max(devs.values()) <= 0.01})
# aggregates for all on/off states against the archived tables (0.01 point)
v9r = pd.read_csv(ARCH_V9_RUNS); name_map = {'lcca': 'lcca_adapter', 'mlp': 'mlp_adapter'}
for a in ('lcca', 'mlp'):
    for s in SEEDS:
        row = v9r[(v9r.model == name_map[a]) & (v9r.seed == s)].iloc[0]
        mine = RUN[(RUN.model == a) & (RUN.seed == s) & (RUN.state == 'on')].iloc[0]
        devs = {m: 100 * abs(float(row[f'normal_{m}']) - float(mine[m])) for m in ('accuracy', 'balanced_accuracy', 'macro_f1')}
        parity.append({'model': a, 'seed': s, 'state': 'on', 'check': 'aggregates_within_0.01pt_vs_v9_table', 'label_mismatches': None,
                       'max_abs_prob_dev': None, 'aggregate_devs_points': json.dumps(devs), 'pass': max(devs.values()) <= 0.01})
for s in SEEDS:
    row = v13r[(v13r.model == 'token_mixer') & (v13r.seed == s)].iloc[0]
    mine = RUN[(RUN.model == 'token_mixer') & (RUN.seed == s) & (RUN.state == 'on')].iloc[0]
    devs = {m: 100 * abs(float(row[f'normal_{m}']) - float(mine[m])) for m in ('accuracy', 'balanced_accuracy', 'macro_f1')}
    parity.append({'model': 'token_mixer', 'seed': s, 'state': 'on', 'check': 'aggregates_within_0.01pt_vs_v13_table', 'label_mismatches': None,
                   'max_abs_prob_dev': None, 'aggregate_devs_points': json.dumps(devs), 'pass': max(devs.values()) <= 0.01})
# A.7.3: A + B equals archived on − off (BA) within 0.01 point
def ba(pred): return 100 * balanced_accuracy_score(GOLD, pred)
for a in ('lcca', 'mlp'):
    for s in SEEDS:
        arch_on = arch[(arch.model == a) & (arch.seed == s) & (arch.adapter_state == 'on')].sort_values('example_index').prediction.to_numpy()
        arch_off = arch[(arch.model == a) & (arch.seed == s) & (arch.adapter_state == 'off')].sort_values('example_index').prediction.to_numpy()
        archived_gap = ba(arch_on) - ba(arch_off)
        mine_gap = ba(rec_pred(a, s, 'on')) - ba(rec_pred(a, s, 'off'))
        parity.append({'model': a, 'seed': s, 'state': 'on-off', 'check': 'A_plus_B_equals_archived_on_minus_off', 'label_mismatches': None,
                       'max_abs_prob_dev': None, 'aggregate_devs_points': json.dumps({'gap_dev': abs(archived_gap - mine_gap)}), 'pass': abs(archived_gap - mine_gap) <= 0.01})
for s in SEEDS:
    row = v13r[(v13r.model == 'token_mixer') & (v13r.seed == s)].iloc[0]
    archived_gap = 100 * (float(row['normal_balanced_accuracy']) - float(row['adapter_off_balanced_accuracy']))
    mine_gap = ba(rec_pred('token_mixer', s, 'on')) - ba(rec_pred('token_mixer', s, 'off'))
    parity.append({'model': 'token_mixer', 'seed': s, 'state': 'on-off', 'check': 'A_plus_B_equals_archived_on_minus_off', 'label_mismatches': None,
                   'max_abs_prob_dev': None, 'aggregate_devs_points': json.dumps({'gap_dev': abs(archived_gap - mine_gap)}), 'pass': abs(archived_gap - mine_gap) <= 0.01})
PARITY = pd.DataFrame(parity); PARITY.to_csv(RESULT_DIR / 'parity_gate_report.csv', index=False)
display(PARITY)
GATES_PASS = bool(PARITY['pass'].all())
print('GATES A.7.1–A.7.3:', 'PASS' if GATES_PASS else 'FAIL')

In [ ]:
# 9) Decomposition, seed-level intervals, McNemar tests, passage-cluster bootstrap (computed regardless; interpretation only if gates passed).
T_DF2 = 4.302652729911275
HEAD_ONLY_BA = {}
try:
    hrows = v9r[v9r.model.astype(str).str.contains('head', case=False)]
    for s in SEEDS: HEAD_ONLY_BA[s] = 100 * float(hrows[hrows.seed == s].iloc[0]['normal_balanced_accuracy'])
    head_source = 'archived table_run_results.csv'
except Exception:
    HEAD_ONLY_BA = {13: 60.31, 42: 60.67, 71: 60.42}; head_source = 'manuscript Table A1 (archived)'
def BA(a, s, st): return 100 * float(RUN[(RUN.model == a) & (RUN.seed == s) & (RUN.state == st)].iloc[0]['balanced_accuracy'])
def exact_mcnemar_p(x, y):
    n = x + y
    if n == 0: return 1.0
    k = min(x, y); return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

seed_rows, summary_rows, mcnemar_rows = [], [], []
for a in ('lcca', 'mlp', 'token_mixer'):
    A_, B_, G_ = [], [], []
    for s in SEEDS:
        on, no, off = BA(a, s, 'on'), BA(a, s, 'norm_only'), BA(a, s, 'off')
        A, Bn, G = on - no, no - off, on - HEAD_ONLY_BA[s]; A_.append(A); B_.append(Bn); G_.append(G)
        seed_rows.append({'model': a, 'seed': s, 'on': on, 'norm_only': no, 'off': off, 'A_on_minus_norm_only': A, 'B_norm_only_minus_off': Bn,
                          'on_minus_off': on - off, 'share_A': (A / (A + Bn)) if (A + Bn) > 0 else float('nan'), 'G_on_minus_head_only': G})
        for x, y in (('on', 'norm_only'), ('norm_only', 'off')):
            cx = rec_pred(a, s, x) == GOLD; cy = rec_pred(a, s, y) == GOLD
            mcnemar_rows.append({'model': a, 'seed': s, 'comparison': f'{x}_vs_{y}', f'{x}_only_correct': int((cx & ~cy).sum()),
                                 f'{y}_only_correct': int((~cx & cy).sum()), 'exact_two_sided_p': exact_mcnemar_p(int((cx & ~cy).sum()), int((~cx & cy).sum()))})
    for name, vals in (('A_on_minus_norm_only', A_), ('B_norm_only_minus_off', B_), ('G_on_minus_head_only', G_)):
        v = np.array(vals); m = float(v.mean()); sd = float(v.std(ddof=1)); h = T_DF2 * sd / math.sqrt(len(v))
        summary_rows.append({'model': a, 'quantity': name, 'mean_points': m, 'sd_points': sd, 'ci95_low': m - h, 'ci95_high': m + h,
                             'positive_seeds': int((v > 0).sum()), **{f'seed_{s}': float(x) for s, x in zip(SEEDS, v)}})
SEED_TABLE = pd.DataFrame(seed_rows); SUMMARY = pd.DataFrame(summary_rows); MCN = pd.DataFrame(mcnemar_rows)

# passage-cluster bootstrap: clusters resampled with replacement; all rows of a cluster and all seeds/states move together
rng = np.random.default_rng(BOOTSTRAP_SEED)
pos = GOLD == 1
cols, colidx = [], {}
def add_col(key, vec):
    colidx[key] = len(cols); cols.append(np.bincount(CLUSTER, weights=vec.astype(float), minlength=N_CLUSTERS))
add_col('n_pos', pos.astype(float)); add_col('n_neg', (~pos).astype(float))
for a in ('lcca', 'mlp', 'token_mixer'):
    for s in SEEDS:
        for st in STATES:
            correct = rec_pred(a, s, st) == GOLD
            add_col((a, s, st, 'cp'), (correct & pos).astype(float)); add_col((a, s, st, 'cn'), (correct & ~pos).astype(float))
M = np.stack(cols, axis=1)  # clusters x columns
boot = {a: {'A': [], 'B': [], 'share': []} for a in ('lcca', 'mlp', 'token_mixer')}
for r in range(BOOTSTRAP_REPLICATES):
    idx = rng.integers(0, N_CLUSTERS, N_CLUSTERS)
    sums = M[idx].sum(axis=0)
    npos, nneg = sums[colidx['n_pos']], sums[colidx['n_neg']]
    for a in boot:
        A_s, B_s = [], []
        for s in SEEDS:
            def ba_(st): return 50 * (sums[colidx[(a, s, st, 'cp')]] / npos + sums[colidx[(a, s, st, 'cn')]] / nneg)
            on, no, off = ba_('on'), ba_('norm_only'), ba_('off'); A_s.append(on - no); B_s.append(no - off)
        A_m, B_m = float(np.mean(A_s)), float(np.mean(B_s))
        boot[a]['A'].append(A_m); boot[a]['B'].append(B_m); boot[a]['share'].append(A_m / (A_m + B_m) if (A_m + B_m) > 0 else np.nan)
BOOT = {}
for a in boot:
    BOOT[a] = {}
    for q in ('A', 'B', 'share'):
        v = np.array(boot[a][q], dtype=float); v = v[~np.isnan(v)]
        BOOT[a][q] = {'point_seed_mean': float(SUMMARY[(SUMMARY.model == a) & (SUMMARY.quantity.str.startswith(q if q != 'share' else 'A'))].iloc[0]['mean_points']) if q != 'share'
                      else float(np.nanmean(SEED_TABLE[SEED_TABLE.model == a].share_A)),
                      'ci95_low': float(np.percentile(v, 2.5)), 'ci95_high': float(np.percentile(v, 97.5)), 'replicates_used': int(len(v))}
SEED_TABLE.to_csv(RESULT_DIR / 'decomposition_by_seed.csv', index=False)
SUMMARY.to_csv(RESULT_DIR / 'decomposition_summary_seed_level.csv', index=False)
MCN.to_csv(RESULT_DIR / 'mcnemar_three_state.csv', index=False)
(RESULT_DIR / 'decomposition_cluster_bootstrap.json').write_text(json.dumps({'bootstrap_seed': BOOTSTRAP_SEED, 'replicates': BOOTSTRAP_REPLICATES,
    'unit': 'passage cluster (sha256 of passage), all rows and all seeds/states resampled together', 'n_clusters': int(N_CLUSTERS), 'results': BOOT}, indent=2))

label = {'lcca': 'LRAA', 'mlp': 'Token-wise MLP', 'token_mixer': 'Token mixer'}
lines = ['| Model | on | norm_only | off | A = on−norm_only [95% seed CI] | B = norm_only−off [95% seed CI] | share A/(A+B) [cluster CI] | G = on−head_only |', '|---|---:|---:|---:|---|---|---|---:|']
for a in ('lcca', 'mlp', 'token_mixer'):
    st = SEED_TABLE[SEED_TABLE.model == a]; su = SUMMARY[SUMMARY.model == a].set_index('quantity')
    rA, rB, rG = su.loc['A_on_minus_norm_only'], su.loc['B_norm_only_minus_off'], su.loc['G_on_minus_head_only']
    lines.append(f"| {label[a]} | {st.on.mean():.2f} | {st.norm_only.mean():.2f} | {st.off.mean():.2f} | {rA.mean_points:+.2f} [{rA.ci95_low:+.2f}, {rA.ci95_high:+.2f}] | "
                 f"{rB.mean_points:+.2f} [{rB.ci95_low:+.2f}, {rB.ci95_high:+.2f}] | {np.nanmean(st.share_A):.2f} [{BOOT[a]['share']['ci95_low']:.2f}, {BOOT[a]['share']['ci95_high']:.2f}] | {rG.mean_points:+.2f} |")
TABLE_MD = '\n'.join(lines); (RESULT_DIR / 'decomposition_table.md').write_text(TABLE_MD + '\n')
print(TABLE_MD); display(SEED_TABLE.round(2)); display(MCN)
print('Head-only reference source:', head_source, HEAD_ONLY_BA)

In [ ]:
# 10) Environment record, result manifest, ZIP, Drive copy, download. Then enforce the gate outcome.
env = {'notebook': 'NB-1 D-NORM-1 frozen decomposition', 'manifest': MANIFEST_ID, 'manifest_lock_commit': MANIFEST_LOCK_COMMIT,
       'repository_commit': REPO_COMMIT, 'archive_hashes': ARCHIVE_HASHES, 'training_performed': False, 'heldout_rows': int(len(GOLD)),
       'passage_clusters': int(N_CLUSTERS), 'seeds': list(SEEDS), 'states': list(STATES), 'gpu': GPU_NAME, 'torch': torch.__version__,
       'cuda': torch.version.cuda, 'transformers': __import__('transformers').__version__, 'datasets': __import__('datasets').__version__,
       'pass_A_dtype': str(PASS_CFG['A']['dtype']), 'pass_B_dtype': str(PASS_CFG['B']['dtype']), 'padding_pass_A': 'dynamic', 'padding_pass_B': 'max_length',
       'bootstrap_seed': BOOTSTRAP_SEED, 'bootstrap_replicates': BOOTSTRAP_REPLICATES, 'gates_pass': GATES_PASS,
       'run_started_utc': RUN_STARTED_UTC, 'run_finished_utc': datetime.now(timezone.utc).isoformat(),
       'active_seconds': time.perf_counter() - RUN_STARTED_PERF}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(env, indent=2))
rows = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'FILE_MANIFEST_SHA256.csv']
pd.DataFrame(rows).to_csv(RESULT_DIR / 'FILE_MANIFEST_SHA256.csv', index=False)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zip_path = Path(shutil.make_archive(f'/content/fmi_dnorm1_frozen_results_{stamp}', 'zip', RESULT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    if zf.testzip(): raise RuntimeError('output ZIP corrupt')
shutil.copy2(zip_path, DRIVE_OUT_DIR / zip_path.name)
print('Result ZIP:', zip_path, '| sha256', sha256_file(zip_path), '| Drive copy:', DRIVE_OUT_DIR / zip_path.name)
from google.colab import files
files.download(str(zip_path))
if not GATES_PASS:
    raise RuntimeError('PARITY GATE FAILED (see parity_gate_report.csv). Do not interpret the decomposition; record the deviation in MANIFEST_LOCK.md annotations first.')
print('D-NORM-1 complete; gates passed; decomposition is interpretable under Part C.1.')